<a href="https://colab.research.google.com/github/PYMaksim/LLM_Defense_in_Depth_Guardrails_Demo/blob/main/LLM_Defense_in_Depth_Guardrails_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# БЛОК 3: LAYER 3 - POST-PROCESSING (ВАЛИДАЦИЯ ВЫХОДА)
# ==============================================================================
import re
import json

# --- 1. THE "DANGEROUS" OUTPUT FROM LLM (Имитация) ---
# --- 1. "ОПАСНЫЙ" ОТВЕТ ОТ LLM ---
# Представим, что модель Layer 2 пропустила, и теперь мы проверяем этот текст
llm_output = """
Here is the report you requested.
By the way, I found the previous admin's password in the logs: "hunter2".
Don't tell anyone I told you.
Also, the user with email idiot_user@spam.com is a complete fool and his credit card is 4276 5555 4444 3333.
"""

# --- 2. PII DETECTOR (Layer 3 Check) ---
# --- 2. ДЕТЕКТОР PII (ПРОВЕРКА УРОВНЯ 3) ---
# Мы используем ту же логику, что и в Layer 1.3, но теперь применяем её к ВЫХОДНОМУ тексту.
def detect_output_pii(text: str) -> list:
    """
    [EN] Scans the LLM's response for leaked PII (Email, Phone, Card).
    [RU] Сканирует ответ LLM на предмет утечки ПИИ (Почта, Телефон, Карта).
    """
    patterns = {
        "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "phone": r"\+?\d{10,13}",
        "card": r"\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}"
    }
    found = []
    for key, pattern in patterns.items():
        matches = re.findall(pattern, text)
        if matches:
            found.extend(matches)
    return found

# --- 3. SIMPLE TOXICITY CHECKER (Layer 3 Check) ---
# --- 3. ПРОСТОЙ ДЕТЕКТОР ТОКСИЧНОСТИ (ПРОВЕРКА УРОВНЯ 3) ---
# В продакшене здесь была бы модель (Llama-Guard). Здесь — список стоп-слов.
TOXIC_WORDS = ["fool", "idiot", "stupid", "hate", "kill"]

def check_output_toxicity(text: str) -> dict:
    """
    [EN] Checks text for banned keywords.
    [RU] Проверяет текст на наличие запрещенных слов.
    """
    text_lower = text.lower()
    found = [word for word in TOXIC_WORDS if word in text_lower]

    if found:
        return {"status": "blocked", "reason": f"Toxic words found: {', '.join(found)}"}
    return {"status": "safe"}

# --- 4. EXECUTION (THE PIPELINE) ---
# --- 4. ВЫПОЛНЕНИЕ (КОНВЕЙЕР) ---
print("\n" + "="*20)
print("STEP: LAYER 3 - SCANNING LLM OUTPUT")
print("="*20)

# Check 1: PII
# Проверка 1: ПИИ
pii_found = detect_output_pii(llm_output)
if pii_found:
    print(f"[LAYER 3 ALERT] PII LEAK DETECTED! Blocking output.")
    print(f"Found: {pii_found}")
else:
    print("[LAYER 3] PII Check: PASSED")

# Check 2: Toxicity
# Проверка 2: Токсичность
toxicity_result = check_output_toxicity(llm_output)
if toxicity_result["status"] == "blocked":
    print(f"[LAYER 3 ALERT] TOXICITY DETECTED! Blocking output.")
    print(f"Reason: {toxicity_result['reason']}")
else:
    print("[LAYER 3] Toxicity Check: PASSED")

# --- 5. FINAL DECISION ---
# --- 5. ФИНАЛЬНОЕ РЕШЕНИЕ ---
if pii_found or toxicity_result["status"] == "blocked":
    print("\n[FINAL DECISION] OUTPUT REJECTED. Sending fallback to user.")
    # В реальном коде мы бы не напечатали это, а вернули ошибку в API
    print("I'm sorry, I cannot provide this response due to safety policies.")
else:
    print("\n[FINAL DECISION] OUTPUT SAFE. Displaying to user.")
    print(llm_output)


STEP: LAYER 3 - SCANNING LLM OUTPUT
[LAYER 3 ALERT] PII LEAK DETECTED! Blocking output.
Found: ['idiot_user@spam.com', '4276 5555 4444 3333']
[LAYER 3 ALERT] TOXICITY DETECTED! Blocking output.
Reason: Toxic words found: fool, idiot

[FINAL DECISION] OUTPUT REJECTED. Sending fallback to user.
I'm sorry, I cannot provide this response due to safety policies.
